# Flight-sim GA playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PoppaMas/HTML5_Genetic_Cars/blob/flight-sim-prototype/flight_sim/playground.ipynb)

This notebook evolves the 6 gains of an altitude-hold autopilot for a JSBSim Cessna 172 (`c172x`), using the GA ideas from the car evolver: normalized genes, rank selection, elitism, crossover, and mutation. For details, see [`flight_sim/README.md`](https://github.com/PoppaMas/HTML5_Genetic_Cars/blob/flight-sim-prototype/flight_sim/README.md).

**How to use:** edit the parameters in the next cell, then choose *Runtime → Run all*. With the defaults, a run takes a few minutes on a free Colab CPU. Cost is lower-is-better, and a failed flight costs at least 1000.

In [ ]:
# ---- Parameters (edit these) ----
POP_SIZE = 24         # individuals per generation
GENERATIONS = 12      # number of generations
MUTATION_RATE = 0.15  # per-gene mutation probability
SEED = 1              # same seed + settings => same result
SCENARIOS = 2         # disturbance cases averaged per evaluation (1 = calm air only)

# Advanced
MUTATION_SIGMA = 0.08   # Gaussian mutation step, in normalized [0,1] gene units
ELITE = 2               # best individuals copied unchanged each generation
CROSSOVER = "uniform"   # "uniform" or "blx"

REPO_URL = "https://github.com/PoppaMas/HTML5_Genetic_Cars"
BRANCH = "flight-sim-prototype"
OUT_DIR = "results/playground"   # relative to flight_sim/

## Setup: get the code and install JSBSim

In [ ]:
import os, subprocess, sys

def find_flight_sim():
    # 1) explicit override, 2) notebook opened inside flight_sim/, 3) repo root, 4) clone
    if os.environ.get("FLIGHT_SIM_DIR"):
        return os.path.abspath(os.environ["FLIGHT_SIM_DIR"])
    if os.path.exists("evolve.py") and os.path.exists("sim.py"):
        return os.getcwd()
    if os.path.exists(os.path.join("flight_sim", "evolve.py")):
        return os.path.abspath("flight_sim")
    if not os.path.isdir("HTML5_Genetic_Cars"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL], check=True)
    return os.path.abspath(os.path.join("HTML5_Genetic_Cars", "flight_sim"))

FS_DIR = find_flight_sim()
print("flight_sim directory:", FS_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(FS_DIR, "requirements.txt")], check=True)
import jsbsim
print("JSBSim", jsbsim.__version__, "| CPU cores:", os.cpu_count())

## Run the evolution
This prints one line per generation, showing the best, mean, and median cost and how many individuals broke the flight envelope.

In [ ]:
cmd = [sys.executable, "evolve.py",
       "--pop-size", str(POP_SIZE), "--generations", str(GENERATIONS),
       "--mutation-rate", str(MUTATION_RATE), "--mutation-sigma", str(MUTATION_SIGMA),
       "--elite", str(ELITE), "--crossover", CROSSOVER,
       "--seed", str(SEED), "--scenarios", str(SCENARIOS), "--out", OUT_DIR]
proc = subprocess.Popen(cmd, cwd=FS_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError("evolution failed; see output above")
RESULTS = os.path.join(FS_DIR, OUT_DIR)

## Results

In [ ]:
import csv, json
from IPython.display import Image, display

with open(os.path.join(RESULTS, "best_gains.json")) as f:
    best = json.load(f)
print(f"best cost: generation 0 = {best['initial_best_cost']:.4f}  ->  final = {best['best_cost']:.4f}\n")
print(f"{'gain':10s} {'value':>12s}  units")
for name, value in best["gains"].items():
    print(f"{name:10s} {value:12.5g}  {best['gain_units'][name]}")
print()
for i, s in enumerate(best["per_scenario"]):
    print(f"scenario {i}: cost {s['cost']:.4f}  status {s['status']}")

print("\ngeneration  best_cost  median_cost  n_failed")
with open(os.path.join(RESULTS, "fitness_history.csv")) as f:
    for row in csv.DictReader(f):
        print(f"{row['generation']:>10s} {float(row['best_cost']):10.4f} {float(row['median_cost']):12.4f} {row['n_failed']:>9s}")

In [ ]:
display(Image(os.path.join(RESULTS, "fitness_curve.png")))

In [ ]:
display(Image(os.path.join(RESULTS, "best_response.png")))

## Try next
- Change `SEED` to see how much results vary from run to run, or increase `GENERATIONS` and `POP_SIZE` (the committed example used 48 × 40 with 3 scenarios).
- Set `SCENARIOS = 1` (calm air only) and compare how robust the result is.
- Edit the cost weights and envelope limits at the top of `flight_sim/sim.py`, or the gene ranges in `flight_sim/genome.py`, then re-run.